# Sesión 05 - Momentos de una variable

Objetivo: calcular momentos, estadísticos de posición y dispersión, y expectativa condicional.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(21)
pd.set_option("display.precision", 4)


## 1. Variable con cola derecha

Simulamos montos de compra con distribución lognormal.


### Lectura matemática

- **Distribución asumida:** Lognormal para montos positivos asimétricos.
- **Parámetros estimados:** media, varianza, asimetría, curtosis y cuantiles.
- **Supuesto que puede fallar:** cola más pesada que lognormal o mezcla de segmentos.
- **Diagnóstico:** media vs mediana, histograma, boxplot y sensibilidad a outliers.


In [ ]:
compras = rng.lognormal(mean=3.1, sigma=0.55, size=5_000)
serie = pd.Series(compras, name="monto")

resumen = pd.Series(
    {
        "media": serie.mean(),
        "mediana": serie.median(),
        "varianza": serie.var(ddof=0),
        "desv_est": serie.std(ddof=0),
        "cv": serie.std(ddof=0) / serie.mean(),
        "asimetria": stats.skew(serie),
        "curtosis_exceso": stats.kurtosis(serie),
        "p10": serie.quantile(0.10),
        "p90": serie.quantile(0.90),
    }
)
resumen


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(serie, bins=50, alpha=0.75)
axes[0].axvline(serie.mean(), color="crimson", label="media")
axes[0].axvline(serie.median(), color="darkgreen", label="mediana")
axes[0].set_title("Montos de compra")
axes[0].legend()

axes[1].boxplot(serie, vert=False)
axes[1].set_title("Boxplot: posición y dispersión")
plt.show()


## 2. Sensibilidad a outliers


In [ ]:
con_outliers = pd.concat([serie, pd.Series([500, 700, 900])], ignore_index=True)
comparacion = pd.DataFrame(
    {
        "sin_outliers": [serie.mean(), serie.median(), serie.std(ddof=0), serie.quantile(0.75) - serie.quantile(0.25)],
        "con_outliers": [con_outliers.mean(), con_outliers.median(), con_outliers.std(ddof=0), con_outliers.quantile(0.75) - con_outliers.quantile(0.25)],
    },
    index=["media", "mediana", "desv_est", "IQR"],
)
comparacion


## 3. Momentos desde una PMF


In [ ]:
x = np.arange(0, 7)
p = stats.binom(n=6, p=0.35).pmf(x)

ex = np.sum(x * p)
ex2 = np.sum((x ** 2) * p)
var = ex2 - ex ** 2

print(f"E[X] = {ex:.3f}")
print(f"E[X^2] = {ex2:.3f}")
print(f"Var(X) = E[X^2] - E[X]^2 = {var:.3f}")


## 4. Expectativa condicional

Simulamos ventas por clima. En regresión, muchas veces buscamos estimar $E[Y|X]$.


### Lectura matemática

- **Objeto estimado:** $E[Y\mid X]$, el mejor predictor bajo pérdida cuadrática.
- **Parámetro estimado:** media condicional por grupo.
- **Supuesto que puede fallar:** confundir diferencia descriptiva con efecto causal.
- **Diagnóstico:** tamaños de grupo, dispersión dentro de grupo e intervalos/boxplots.


In [ ]:
n = 3_000
clima = rng.choice(["soleado", "nublado", "lluvia"], size=n, p=[0.45, 0.35, 0.20])
media_por_clima = {"soleado": 120, "nublado": 95, "lluvia": 70}
ventas = np.array([rng.normal(media_por_clima[c], 18) for c in clima])
ventas = np.clip(ventas, 0, None)

df = pd.DataFrame({"clima": clima, "ventas": ventas})
condicional = df.groupby("clima")["ventas"].agg(["count", "mean", "median", "std"])
condicional


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column="ventas", by="clima", ax=ax)
ax.set_title("Distribución de ventas por clima")
ax.figure.suptitle("")
ax.set_xlabel("clima")
ax.set_ylabel("ventas")
plt.show()


## 5. Momentos en el dataset de demanda

Este bloque recupera el caso real del material previo: momentos, percentiles y expectativa condicional sobre demanda, precio, promociones y clima.


In [ ]:
from pathlib import Path

def display(obj):
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

def encontrar_data_dir():
    for candidato in [Path("data_sources"), Path("../data_sources")]:
        if candidato.exists():
            return candidato
    return None

DATA_DIR = encontrar_data_dir()
if DATA_DIR is None or not (DATA_DIR / "sales_data.csv").exists():
    print("No se encontró data_sources/sales_data.csv. Se mantiene la sección sintética.")
else:
    cols = ["Demand", "Units Sold", "Price", "Discount", "Weather Condition", "Promotion", "Category"]
    ventas_df = pd.read_csv(DATA_DIR / "sales_data.csv", usecols=cols)
    numericas = ["Demand", "Units Sold", "Price", "Discount"]
    momentos = ventas_df[numericas].agg(["mean", "median", "var", "std", "skew"]).T
    momentos["cv"] = momentos["std"] / momentos["mean"]
    display(momentos)


In [ ]:
if DATA_DIR is not None and (DATA_DIR / "sales_data.csv").exists():
    expectativa_condicional = (
        ventas_df.groupby(["Weather Condition", "Promotion"])["Demand"]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean", ascending=False)
    )
    display(expectativa_condicional.head(12))


## Práctica

Cambia las medias por clima o agrega una variable `campaña`. Estima $E[ventas|clima]$ y $E[ventas|clima,campaña]$.
